# Lab Day 2 — Backbone, công thức huấn luyện và suy luận trên DeepWeeds

Notebook **khung** cho Colab hoặc Kaggle. Các ô có chữ `TODO` là phần **bạn tự viết**; ô cài đặt, tải dữ liệu
và chạy `eval.py` đã viết sẵn. Đọc `README.md` (quy tắc chia dữ liệu, cách đánh giá), `GUIDE.md` và `RUBRIC.md` trước.

**Quy tắc quan trọng nhất:** chọn mọi thứ trên **val**; **test chỉ chạy một lần mỗi seed ở Bước 4**.

Bản đồ: Bước 0 (EDA, kiểm tra) → Bước 1 (backbone) → Bước 2 (công thức huấn luyện) → Bước 3 (suy luận)
→ Bước 4 (chung kết + `eval.py`) → Bước 5 (xlsx, biểu đồ, báo cáo).

## 0. Cài đặt môi trường

In [4]:
# Clone the student repository
!git clone https://github.com/tranphat1506/K4-Track4-Day2-Le-Chau-Tran-Phat-2A202602545-Deeplearning-Advance

# Update REPO_DIR variable to match the newly cloned repository name
REPO_DIR = "K4-Track4-Day2-Le-Chau-Tran-Phat-2A202602545-Deeplearning-Advance"

Cloning into 'K4-Track4-Day2-Le-Chau-Tran-Phat-2A202602545-Deeplearning-Advance'...
remote: Enumerating objects: 44, done.
remote: Counting objects: 100% (10/10), done.
remote: Compressing objects: 100% (10/10), done.
remote: Total 44 (delta 0), reused 9 (delta 0), pack-reused 34 (from 1)
Receiving objects: 100% (44/44), 78.85 KiB | 667.00 KiB/s, done.
Resolving deltas: 100% (7/7), done.


In [13]:
import sys
import importlib

# Xóa 'starter' ra khỏi đường dẫn nếu nó đang tồn tại
sys.path = [p for p in sys.path if not p.endswith('starter')]

# Cập nhật lại đường dẫn mới vào thư mục code
sys.path.insert(0, f"{REPO_DIR}/code")

# Ép tải lại các module
if 'dataset' in sys.modules:
    import dataset
    importlib.reload(dataset)

In [14]:
# Cài thư viện (Colab/Kaggle). Kaggle: bật Internet trong Settings.
!pip -q install timm openpyxl

import sys, os, platform
import numpy as np, pandas as pd, torch, timm

sys.path.insert(0, REPO_DIR)                 # để import eval.py
sys.path.insert(0, f"{REPO_DIR}/code")    # hoặc thư mục code/ của bạn sau khi chép starter vào

print("python", platform.python_version(), "| torch", torch.__version__, "| timm", timm.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG CÓ GPU")

python 3.13.15 | torch 2.11.0+cu130 | timm 1.0.30
GPU: Tesla T4


### Tải dữ liệu (đã viết sẵn)
Ảnh từ Zenodo (~490 MB, kiểm tra MD5), nhãn và các file chia fold từ GitHub của tác giả.
**Sau khi giải nén, hãy `ls` để xem ảnh nằm ở thư mục nào** rồi đặt `IMAGES_DIR` cho đúng.

In [2]:
import hashlib

os.makedirs("data/labels", exist_ok=True)
if not os.path.exists("data/images.zip"):
    !wget -q -O data/images.zip "https://zenodo.org/records/7939060/files/images.zip?download=1"

h = hashlib.md5()
with open("data/images.zip", "rb") as f:
    for chunk in iter(lambda: f.read(1 << 20), b""):
        h.update(chunk)
assert h.hexdigest() == "b7b30f96d466fba86016aa5a26606e0f", f"MD5 sai: {h.hexdigest()}"

!unzip -q -n data/images.zip -d data/
!ls data | head

BASE = "https://raw.githubusercontent.com/AlexOlsen/DeepWeeds/master/labels"
for name in ["labels", "train_subset0", "val_subset0", "test_subset0"]:
    !wget -q -O data/labels/{name}.csv {BASE}/{name}.csv
!ls -la data/labels

20160928-140314-0.jpg
20160928-140337-0.jpg
20160928-140731-0.jpg
20160928-140747-0.jpg
20160928-141107-0.jpg
20160928-141135-0.jpg
20160928-141355-0.jpg
20160928-141421-0.jpg
20160928-141437-0.jpg
20160928-142056-0.jpg
total 1812
drwxr-xr-x 2 root root   4096 Oct  6 08:41 .
drwxr-xr-x 3 root root 819200 Oct  6 08:41 ..
-rw-r--r-- 1 root root 598898 Oct  6 08:41 labels.csv
-rw-r--r-- 1 root root  84183 Oct  6 08:41 test_subset0.csv
-rw-r--r-- 1 root root 252039 Oct  6 08:41 train_subset0.csv
-rw-r--r-- 1 root root  84039 Oct  6 08:41 val_subset0.csv


In [16]:
import os
import shutil
from pathlib import Path

# Tạo thư mục data/images nếu chưa tồn tại
target_dir = Path("data/images")
target_dir.mkdir(parents=True, exist_ok=True)

# Tìm và di chuyển tất cả các file ảnh .jpg từ data/ sang data/images/
source_dir = Path("data")
moved_count = 0

for file_path in source_dir.glob("*.jpg"):
    shutil.move(str(file_path), str(target_dir / file_path.name))
    moved_count += 1

print(f"Đã di chuyển thành công {moved_count} ảnh vào thư mục {target_dir}")

Đã di chuyển thành công 17509 ảnh vào thư mục data/images


In [6]:
IMAGES_DIR = "data/images"   # TODO: kiểm tra đúng thư mục chứa các file .jpg sau khi giải nén
LABELS_DIR = "data/labels"

## Bước 0 — EDA và kiểm tra pipeline
Mục tiêu: hiểu dữ liệu, chạy các kiểm tra chia dữ liệu (README mục 2.1), kiểm tra pipeline trước khi chạy thật.

In [15]:
# Bước 0: EDA và kiểm tra pipeline
from dataset import load_split, check_split
import matplotlib.pyplot as plt
import seaborn as sns

train_df, val_df, test_df = load_split(LABELS_DIR, fold=0)
stats = check_split(train_df, val_df, test_df, IMAGES_DIR)

# Vẽ biểu đồ phân bố
plt.figure(figsize=(10, 5))
sns.barplot(x=list(stats['train_per_class'].keys()), y=list(stats['train_per_class'].values()))
plt.title('Phân bố các lớp trong tập Train')
plt.show()


Total: 17509 (Train: 10501, Val: 3501, Test: 3507)


AssertionError: File 20171219-112533-3.jpg not found in data/images

In [ ]:
# Bước 0.3: Kiểm tra pipeline
from train import set_seed
import torch
import torch.nn as nn
from model import build_model

set_seed(42)
# Kiểm tra loss ban đầu xấp xỉ -ln(1/9) = 2.197
dummy_model = build_model("resnet50", pretrained=False, num_classes=9)
dummy_model.eval()
dummy_logits = dummy_model(torch.randn(2, 3, 224, 224))
dummy_loss = nn.CrossEntropyLoss()(dummy_logits, torch.tensor([0, 1]))
print(f"Loss ban đầu: {dummy_loss.item():.4f} (Kỳ vọng ~ 2.197)")


## Bước 1 — So sánh backbone (≥ 5)
Cùng công thức nền `T00` (GUIDE.md mục 1.4), cùng seed, mỗi backbone một lần chạy. Mã thí nghiệm `B01`, `B02`, ...

In [ ]:
from train import Config, run
import pandas as pd

backbones_to_test = ['resnet50', 'resnext50', 'swin_tiny', 'efficientnet_b0', 'mobilenetv3']
results = []

for i, bb in enumerate(backbones_to_test):
    cfg = Config(exp_id=f"B{i:02d}", backbone=bb, seed=42, epochs=5) # Giảm epochs để test nhanh
    print(f"\n--- Đang train {bb} ---")
    res = run(cfg)
    results.append({
        'backbone': bb,
        'F1_val': res['val_macro_f1'],
        'params_M': res['params_m'],
        'GMACs': res['gmacs'],
        'Time_per_epoch': res['time_per_epoch']
    })

df_results = pd.DataFrame(results)
display(df_results)


## Bước 2 — Công thức huấn luyện (≥ 3 trục)
Mỗi lần chạy chỉ khác `T00` **một yếu tố**. Mã `T01`, `T02`, ... Ghi rõ khác ở điểm nào.

In [ ]:
from train import Config, run

# Ví dụ: Thử nghiệm đổi hàm Loss sang Focal Loss với backbone tốt nhất (giả sử là resnet50)
cfg_focal = Config(exp_id="T01_Focal", backbone="resnet50", loss="focal", seed=42, epochs=5)
res_focal = run(cfg_focal)
print("Focal Loss F1:", res_focal['val_macro_f1'])

# Thử nghiệm đổi Augmentation sang RandAugment
cfg_randaug = Config(exp_id="T02_RandAug", backbone="resnet50", aug="randaug", seed=42, epochs=5)
res_randaug = run(cfg_randaug)
print("RandAugment F1:", res_randaug['val_macro_f1'])


## Bước 3 — Suy luận (≥ 4 phương pháp) và độ trễ
Làm trên **val**. Không huấn luyện lại. Mã `I00` (1 view, mốc), `I01`, ...

In [ ]:
from model import build_model
from inference import fuse_conv_bn
from benchmark import tta_latency
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
dummy_model = build_model("resnet50", pretrained=False, num_classes=9).to(device)

# Thử nghiệm fuse BatchNorm và đo độ trễ của Test-Time Augmentation (TTA)
print("Đang đo độ trễ TTA với 2 views (Lật ngang)...")
fused_model = fuse_conv_bn(dummy_model)
lat_report = tta_latency(fused_model, k_views=2, img_size=224, device=device)
print("Báo cáo độ trễ:", lat_report)


## Bước 4 — Chung kết: ≥ 3 seed, test một lần mỗi seed
1. Chốt cấu hình trên **val**.
2. Chạy cấu hình cuối **và mốc** (`T00` + `I00`) với cùng số seed, bật `save_test_predictions=True`.
3. Dự đoán lưu ở `predictions/<exp_id>_seed<k>_test.csv`. Rồi chạy `eval.py` (ô dưới).

In [ ]:
from train import Config, run

# Chạy vòng chung kết với 3 seed khác nhau (cấu hình tốt nhất giả định là T02_RandAug)
for seed in [0, 1, 2]:
    # Baseline
    run(Config(exp_id="T00", backbone="resnet50", seed=seed, save_test_predictions=True))
    # Cấu hình Final (ví dụ)
    run(Config(exp_id="F01", backbone="resnet50", aug="randaug", seed=seed, save_test_predictions=True))


In [ ]:
# Tính chỉ số theo đúng định nghĩa của lớp (đã viết sẵn, không sửa eval.py):
!python {REPO_DIR}/eval.py score --pred "predictions/F01_seed*_test.csv" \
    --test-csv {LABELS_DIR}/test_subset0.csv --labels {LABELS_DIR}/labels.csv --tag F01 --out eval_out
!python {REPO_DIR}/eval.py score --pred "predictions/T00_seed*_test.csv" \
    --test-csv {LABELS_DIR}/test_subset0.csv --labels {LABELS_DIR}/labels.csv --tag T00 --out eval_out

In [ ]:
# Tự chấm phần I của RUBRIC (đề xuất, giảng viên xác nhận). Thêm --uncal, --final-val, --latency-p95-ms nếu có.
!python {REPO_DIR}/eval.py grade --final "predictions/F01_seed*_test.csv" \
    --baseline "predictions/T00_seed*_test.csv" \
    --test-csv {LABELS_DIR}/test_subset0.csv --labels {LABELS_DIR}/labels.csv --out eval_out

## Bước 5 — Sản phẩm
`results.xlsx` (các sheet ở GUIDE.md mục 6.1), `curves/<exp_id>_*.png`, `report.md`, `predictions/`, `code/`.

In [ ]:
import pandas as pd
import os

results = []
if os.path.exists("runs"):
    for exp in os.listdir("runs"):
        exp_dir = os.path.join("runs", exp)
        if os.path.isdir(exp_dir):
            for seed_dir in os.listdir(exp_dir):
                hist_path = os.path.join(exp_dir, seed_dir, "history.csv")
                if os.path.exists(hist_path):
                    df = pd.read_csv(hist_path)
                    best_row = df.loc[df['val_macro_f1'].idxmax()]
                    results.append({
                        "exp_id": exp,
                        "seed": seed_dir,
                        "best_epoch": int(best_row['epoch']),
                        "val_macro_f1": best_row['val_macro_f1'],
                        "train_time_per_epoch": best_row['time']
                    })

if results:
    df_results = pd.DataFrame(results)
    display(df_results)
    with pd.ExcelWriter("results.xlsx") as writer:
        df_results.to_excel(writer, sheet_name="Summary", index=False)
    print("\nĐã lưu tổng hợp vào results.xlsx")
else:
    print("Chưa có kết quả train nào trong thư mục runs/ để xuất excel.")
